In [0]:

-- Databricks notebook
-- Purpose: Create and load the Gold dimensional model for SUSEP insurance analytics.
-- Source:
--   - susep_silver.vw_fato_seguro_mensal_susep
-- Targets:
--   - susep_gold.tb_dim_periodo
--   - susep_gold.tb_dim_empresa_susep
--   - susep_gold.tb_dim_ramo_susep
--   - susep_gold.tb_fato_seguro_mensal_susep
--
-- Dimensional model:
--   tb_dim_periodo           1 --- N  tb_fato_seguro_mensal_susep
--   tb_dim_empresa_susep     1 --- N  tb_fato_seguro_mensal_susep
--   tb_dim_ramo_susep        1 --- N  tb_fato_seguro_mensal_susep
--
-- Grain:
--   - Fact table: one row per reference month, SUSEP entity and treated SUSEP branch.
--   - Period dimension: one row per reference month.
--   - Company dimension: one row per SUSEP entity and SCD version.
--   - Branch dimension: one row per SUSEP branch/treatment and SCD version.
--
-- Processing strategy:
--   - Recreate table structures with CREATE OR REPLACE TABLE.
--   - Rebuild all dimensions and the fact table with INSERT OVERWRITE.
--   - Use deterministic surrogate keys based on business keys and SCD attributes.
--   - Keep only analytical/business columns in Gold; Bronze/Silver operational metadata is intentionally not carried forward.
--
-- Naming convention:
--   - Tables start with tb_.
--   - Surrogate keys start with sk_.
--   - Codes start with cd_.
--   - Names start with nm_.
--   - Descriptions start with ds_.
--   - Dates start with dt_.
--   - Numbers start with nr_.
--   - Flags start with fl_.
--   - Values start with vl_.
--   - Quantities start with qt_.

-- ============================================================
-- 1. CONFIGURATION
-- ============================================================

CREATE SCHEMA IF NOT EXISTS susep_gold;

SELECT
  'susep_silver.vw_fato_seguro_mensal_susep' AS source_table,
  'susep_gold.tb_dim_periodo' AS target_period_dimension,
  'susep_gold.tb_dim_empresa_susep' AS target_company_dimension,
  'susep_gold.tb_dim_ramo_susep' AS target_branch_dimension,
  'susep_gold.tb_fato_seguro_mensal_susep' AS target_fact_table,
  CURRENT_TIMESTAMP() AS execution_timestamp;


In [0]:

-- ============================================================
-- 2. CREATE GOLD DELTA TABLE STRUCTURES
-- ============================================================
-- This step recreates the Gold dimensional tables with the expected schema,
-- column comments, Delta properties and partitioning strategy.
--
-- Important:
-- CREATE OR REPLACE TABLE is intentionally used because this notebook is
-- deterministic and the following steps use INSERT OVERWRITE.
--
-- Business note:
-- The SCD columns are kept because they are part of the dimensional design.
-- Operational metadata from Bronze/Silver is not included in Gold.

CREATE OR REPLACE TABLE susep_gold.tb_dim_periodo (
  sk_periodo INT COMMENT 'Surrogate key determinística do período no formato AAAAMM.',
  nr_ano_mes_referencia INT COMMENT 'Competência de referência no formato AAAAMM.',
  nr_ano_referencia INT COMMENT 'Ano da competência de referência.',
  nr_mes_referencia INT COMMENT 'Mês da competência de referência.',
  dt_referencia DATE COMMENT 'Data de referência da competência, sempre no primeiro dia do mês.',
  nm_mes_referencia STRING COMMENT 'Nome do mês da competência em português do Brasil.',
  sg_mes_referencia STRING COMMENT 'Abreviação do mês da competência em português do Brasil.',
  nr_trimestre_referencia INT COMMENT 'Número do trimestre da competência de referência.',
  nm_trimestre_referencia STRING COMMENT 'Descrição do trimestre da competência de referência.',
  nr_semestre_referencia INT COMMENT 'Número do semestre da competência de referência.',
  nm_semestre_referencia STRING COMMENT 'Descrição do semestre da competência de referência.'
)
USING DELTA
COMMENT 'Dimensão de período mensal para análise dos dados públicos de seguros da SUSEP.'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'quality' = 'gold',
  'dominio' = 'seguros_susep',
  'projeto' = 'lakehouse_susep',
  'tipo_tabela' = 'dimensao',
  'granularidade' = 'mes_referencia'
);

CREATE OR REPLACE TABLE susep_gold.tb_dim_empresa_susep (
  sk_empresa_susep BIGINT COMMENT 'Surrogate key determinística da empresa/entidade SUSEP considerando a versão SCD.',
  cd_entidade_susep STRING COMMENT 'Código SUSEP da entidade/seguradora.',
  nm_entidade_susep STRING COMMENT 'Nome da entidade/seguradora conforme base Silver.',
  cd_grupo_empresa_susep_original STRING COMMENT 'Código original do grupo econômico informado na base SES seguros.',
  cd_grupo_empresa_susep_mapeado STRING COMMENT 'Código de grupo econômico após regra de de/para/deduplicação por entidade.',
  nm_grupo_empresa_susep_mapeado STRING COMMENT 'Nome do grupo econômico mapeado para a entidade.',
  nm_empresa_consolidada_original STRING COMMENT 'Nome consolidado originalmente identificado no mapeamento.',
  nm_grupo_concorrencia_n1 STRING COMMENT 'Grupo concorrencial de primeiro nível.',
  nm_grupo_concorrencia_n2 STRING COMMENT 'Grupo concorrencial de segundo nível.',
  ds_tipo_grupo STRING COMMENT 'Descrição do tipo de grupo/empresa para análise concorrencial.',
  ds_regra_mapeamento_empresa STRING COMMENT 'Regra aplicada para consolidar a entidade no grupo concorrencial.',
  ds_confianca_mapeamento_empresa STRING COMMENT 'Nível de confiança do mapeamento de empresa.',
  ds_observacao_mapeamento_empresa STRING COMMENT 'Observação adicional sobre o mapeamento da empresa.',
  fl_alterou_mapeamento_empresa BOOLEAN COMMENT 'Indica se o mapeamento alterou o resultado originalmente consultado.',
  dt_inicio_vigencia_empresa DATE COMMENT 'Data de início de vigência da versão SCD da empresa.',
  dt_fim_vigencia_empresa DATE COMMENT 'Data de fim de vigência da versão SCD da empresa. Usa 9999-12-31 para versão em aberto.',
  fl_registro_atual BOOLEAN COMMENT 'Indica se a versão SCD da empresa é a versão atual.',
  nr_versao_empresa STRING COMMENT 'Número ou código da versão do mapeamento SCD da empresa.'
)
USING DELTA
COMMENT 'Dimensão SCD de empresas e grupos concorrenciais para análise de seguros SUSEP.'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'quality' = 'gold',
  'dominio' = 'seguros_susep',
  'projeto' = 'lakehouse_susep',
  'tipo_tabela' = 'dimensao_scd',
  'granularidade' = 'entidade_susep_versao_scd'
);

CREATE OR REPLACE TABLE susep_gold.tb_dim_ramo_susep (
  sk_ramo_susep BIGINT COMMENT 'Surrogate key determinística do ramo SUSEP considerando tratamento analítico e versão SCD.',
  cd_grupo_ramo_susep STRING COMMENT 'Código do grupo de ramos/produtos SUSEP.',
  nm_grupo_ramo_susep STRING COMMENT 'Nome do grupo de ramos/produtos SUSEP.',
  cd_ramo_susep STRING COMMENT 'Código do ramo SUSEP normalizado em 4 dígitos.',
  nm_ramo_susep STRING COMMENT 'Nome do ramo SUSEP.',
  cd_ramo_tratado STRING COMMENT 'Código tratado/agrupado do ramo para análise executiva.',
  nm_ramo_tratado STRING COMMENT 'Nome tratado/agrupado do ramo para análise executiva.',
  ds_categoria_mercado_susep STRING COMMENT 'Categoria de mercado atribuída ao ramo SUSEP.',
  fl_base_analitica_seguros BOOLEAN COMMENT 'Indica se o ramo compõe a base analítica principal de seguros.',
  fl_ramo_dpvat BOOLEAN COMMENT 'Indica se o ramo pertence ao DPVAT.',
  fl_ramo_acumulacao BOOLEAN COMMENT 'Indica se o ramo pertence a VGBL, sobrevivência ou outro produto de acumulação.',
  fl_ramo_previdencia BOOLEAN COMMENT 'Indica se o ramo pertence a previdência, EFPC ou microsseguro previdenciário.',
  fl_ramo_saude BOOLEAN COMMENT 'Indica se o ramo pertence a saúde suplementar ou saúde resseguro.',
  ds_criterio_classificacao_mercado STRING COMMENT 'Descrição da regra utilizada para classificar o ramo na categoria de mercado SUSEP.',
  ds_justificativa_tratamento_ramo STRING COMMENT 'Justificativa do tratamento ou agrupamento aplicado ao ramo.',
  ds_fonte_regra_mapeamento_ramo STRING COMMENT 'Fonte ou racional da regra de mapeamento do ramo.',
  dt_inicio_vigencia_ramo DATE COMMENT 'Data de início de vigência da versão SCD do ramo.',
  dt_fim_vigencia_ramo DATE COMMENT 'Data de fim de vigência da versão SCD do ramo. Usa 9999-12-31 para versão em aberto.',
  fl_registro_atual BOOLEAN COMMENT 'Indica se a versão SCD do ramo é a versão atual.',
  nr_versao_ramo STRING COMMENT 'Número ou código da versão do mapeamento SCD do ramo.'
)
USING DELTA
COMMENT 'Dimensão SCD de ramos SUSEP, ramos tratados e categorias de mercado para análise de seguros.'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'quality' = 'gold',
  'dominio' = 'seguros_susep',
  'projeto' = 'lakehouse_susep',
  'tipo_tabela' = 'dimensao_scd',
  'granularidade' = 'ramo_susep_tratado_versao_scd'
);

CREATE OR REPLACE TABLE susep_gold.tb_fato_seguro_mensal_susep (
  sk_periodo INT COMMENT 'Surrogate key da dimensão de período.',
  sk_empresa_susep BIGINT COMMENT 'Surrogate key da dimensão de empresa SUSEP.',
  sk_ramo_susep BIGINT COMMENT 'Surrogate key da dimensão de ramo SUSEP.',
  vl_premio_direto DECIMAL(38,2) COMMENT 'Valor de prêmio direto.',
  vl_premio_de_seguros DECIMAL(38,2) COMMENT 'Valor de prêmio de seguros.',
  vl_premio_retido DECIMAL(38,2) COMMENT 'Valor de prêmio retido.',
  vl_premio_ganho DECIMAL(38,2) COMMENT 'Valor de prêmio ganho.',
  vl_sinistro_direto DECIMAL(38,2) COMMENT 'Valor de sinistro direto.',
  vl_sinistro_retido DECIMAL(38,2) COMMENT 'Valor de sinistro retido.',
  vl_despesa_comercializacao DECIMAL(38,2) COMMENT 'Valor de despesa de comercialização/comissão.',
  vl_premio_emitido DECIMAL(38,2) COMMENT 'Valor de prêmio emitido.',
  vl_premio_emitido_capitalizacao DECIMAL(38,2) COMMENT 'Valor de prêmio emitido associado a capitalização quando presente na base.',
  vl_despesa_resseguro DECIMAL(38,2) COMMENT 'Valor de despesa com resseguro.',
  vl_sinistro_ocorrido DECIMAL(38,2) COMMENT 'Valor de sinistro ocorrido.',
  vl_receita_resseguro DECIMAL(38,2) COMMENT 'Valor de receita de resseguro.',
  vl_sinistro_ocorrido_capitalizacao DECIMAL(38,2) COMMENT 'Valor de sinistros ocorridos associados a capitalização quando presente na base.',
  vl_recuperacao_sinistro_ocorrido_capitalizacao DECIMAL(38,2) COMMENT 'Valor de recuperação de sinistros ocorridos associados a capitalização quando presente na base.',
  vl_rvne DECIMAL(38,2) COMMENT 'Valor de RVNE informado na base.',
  vl_convenio_dpvat DECIMAL(38,2) COMMENT 'Valor relacionado ao convênio DPVAT.',
  vl_consorcio_fundos DECIMAL(38,2) COMMENT 'Valor relacionado a consórcios e fundos.'
)
USING DELTA
PARTITIONED BY (sk_periodo)
COMMENT 'Fato mensal de seguros SUSEP com medidas financeiras agregadas por período, empresa e ramo.'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'quality' = 'gold',
  'dominio' = 'seguros_susep',
  'projeto' = 'lakehouse_susep',
  'tipo_tabela' = 'fato',
  'granularidade' = 'periodo_empresa_ramo'
);


In [0]:

-- ============================================================
-- 3. SOURCE TABLE VALIDATION
-- ============================================================
-- Quick validation before loading Gold.

SELECT
  'susep_silver.vw_fato_seguro_mensal_susep' AS source_table,
  COUNT(*) AS qt_linhas,
  COUNT_IF(nr_ano_mes_referencia IS NULL) AS qt_sem_periodo,
  COUNT_IF(cd_entidade_susep IS NULL) AS qt_sem_entidade,
  COUNT_IF(cd_ramo_susep IS NULL AND cd_ramo_tratado IS NULL) AS qt_sem_ramo,
  MIN(dt_referencia) AS dt_referencia_min,
  MAX(dt_referencia) AS dt_referencia_max
FROM susep_silver.vw_fato_seguro_mensal_susep;

SELECT
  'cardinalidade_base_gold' AS ds_validacao,
  COUNT(DISTINCT nr_ano_mes_referencia) AS qt_periodos,
  COUNT(DISTINCT cd_entidade_susep) AS qt_entidades_susep,
  COUNT(DISTINCT cd_ramo_tratado) AS qt_ramos_tratados,
  COUNT(DISTINCT CONCAT_WS('|', CAST(nr_ano_mes_referencia AS STRING), cd_entidade_susep, cd_ramo_tratado)) AS qt_chaves_fato_esperadas
FROM susep_silver.vw_fato_seguro_mensal_susep;


In [0]:

-- ============================================================
-- 4. PREPARE NORMALIZED GOLD SOURCE VIEW
-- ============================================================
-- This temporary view centralizes datatype treatment, code normalization,
-- boolean standardization and SCD date treatment used by all Gold loads.

CREATE OR REPLACE TEMP VIEW vw_gold_base_seguro_mensal AS
WITH src AS (
  SELECT *
  FROM susep_silver.vw_fato_seguro_mensal_susep
), typed AS (
  SELECT
    TRY_CAST(REGEXP_REPLACE(NULLIF(TRIM(CAST(nr_ano_mes_referencia AS STRING)), 'null'), '\\.0$', '') AS INT) AS nr_ano_mes_referencia,
    TRY_CAST(REGEXP_REPLACE(NULLIF(TRIM(CAST(nr_ano_referencia AS STRING)), 'null'), '\\.0$', '') AS INT) AS nr_ano_referencia,
    TRY_CAST(REGEXP_REPLACE(NULLIF(TRIM(CAST(nr_mes_referencia AS STRING)), 'null'), '\\.0$', '') AS INT) AS nr_mes_referencia,
    CASE
      WHEN TRY_CAST(dt_referencia AS DATE) IS NOT NULL THEN TRY_CAST(dt_referencia AS DATE)
      WHEN TRY_CAST(REGEXP_REPLACE(NULLIF(TRIM(CAST(dt_referencia AS STRING)), 'null'), '\\.0$', '') AS INT) IS NOT NULL
        THEN DATE_ADD(DATE '1899-12-30', TRY_CAST(REGEXP_REPLACE(NULLIF(TRIM(CAST(dt_referencia AS STRING)), 'null'), '\\.0$', '') AS INT))
      ELSE NULL
    END AS dt_referencia,

    NULLIF(TRIM(CAST(cd_entidade_susep AS STRING)), 'null') AS cd_entidade_susep,
    NULLIF(TRIM(CAST(nm_entidade_susep AS STRING)), 'null') AS nm_entidade_susep,
    NULLIF(TRIM(CAST(cd_grupo_empresa_susep_original AS STRING)), 'null') AS cd_grupo_empresa_susep_original,
    NULLIF(TRIM(CAST(cd_grupo_empresa_susep_mapeado AS STRING)), 'null') AS cd_grupo_empresa_susep_mapeado,
    NULLIF(TRIM(CAST(nm_grupo_empresa_susep_mapeado AS STRING)), 'null') AS nm_grupo_empresa_susep_mapeado,

    CASE
      WHEN NULLIF(TRIM(CAST(cd_grupo_ramo_susep AS STRING)), 'null') IS NULL THEN NULL
      WHEN TRY_CAST(REGEXP_REPLACE(TRIM(CAST(cd_grupo_ramo_susep AS STRING)), '\\.0$', '') AS INT) IS NOT NULL
        THEN LPAD(CAST(TRY_CAST(REGEXP_REPLACE(TRIM(CAST(cd_grupo_ramo_susep AS STRING)), '\\.0$', '') AS INT) AS STRING), 2, '0')
      ELSE NULLIF(TRIM(CAST(cd_grupo_ramo_susep AS STRING)), 'null')
    END AS cd_grupo_ramo_susep,
    NULLIF(TRIM(CAST(nm_grupo_ramo_susep AS STRING)), 'null') AS nm_grupo_ramo_susep,
    CASE
      WHEN NULLIF(TRIM(CAST(cd_ramo_susep AS STRING)), 'null') IS NULL THEN NULL
      WHEN TRY_CAST(REGEXP_REPLACE(TRIM(CAST(cd_ramo_susep AS STRING)), '\\.0$', '') AS INT) IS NOT NULL
        THEN LPAD(CAST(TRY_CAST(REGEXP_REPLACE(TRIM(CAST(cd_ramo_susep AS STRING)), '\\.0$', '') AS INT) AS STRING), 4, '0')
      ELSE NULLIF(TRIM(CAST(cd_ramo_susep AS STRING)), 'null')
    END AS cd_ramo_susep,
    NULLIF(TRIM(CAST(nm_ramo_susep AS STRING)), 'null') AS nm_ramo_susep,
    CASE
      WHEN NULLIF(TRIM(CAST(cd_ramo_tratado AS STRING)), 'null') IS NULL THEN NULL
      WHEN TRY_CAST(REGEXP_REPLACE(TRIM(CAST(cd_ramo_tratado AS STRING)), '\\.0$', '') AS INT) IS NOT NULL
        THEN LPAD(CAST(TRY_CAST(REGEXP_REPLACE(TRIM(CAST(cd_ramo_tratado AS STRING)), '\\.0$', '') AS INT) AS STRING), 4, '0')
      ELSE NULLIF(TRIM(CAST(cd_ramo_tratado AS STRING)), 'null')
    END AS cd_ramo_tratado,
    NULLIF(TRIM(CAST(nm_ramo_tratado AS STRING)), 'null') AS nm_ramo_tratado,

    NULLIF(TRIM(CAST(ds_categoria_mercado_susep AS STRING)), 'null') AS ds_categoria_mercado_susep,
    CASE
      WHEN LOWER(NULLIF(TRIM(CAST(fl_base_analitica_seguros AS STRING)), 'null')) IN ('true', '1', 'sim', 's', 'yes', 'y') THEN TRUE
      WHEN LOWER(NULLIF(TRIM(CAST(fl_base_analitica_seguros AS STRING)), 'null')) IN ('false', '0', 'nao', 'não', 'n', 'no') THEN FALSE
      ELSE FALSE
    END AS fl_base_analitica_seguros,
    CASE
      WHEN LOWER(NULLIF(TRIM(CAST(fl_ramo_dpvat AS STRING)), 'null')) IN ('true', '1', 'sim', 's', 'yes', 'y') THEN TRUE
      ELSE FALSE
    END AS fl_ramo_dpvat,
    CASE
      WHEN LOWER(NULLIF(TRIM(CAST(fl_ramo_acumulacao AS STRING)), 'null')) IN ('true', '1', 'sim', 's', 'yes', 'y') THEN TRUE
      ELSE FALSE
    END AS fl_ramo_acumulacao,
    CASE
      WHEN LOWER(NULLIF(TRIM(CAST(fl_ramo_previdencia AS STRING)), 'null')) IN ('true', '1', 'sim', 's', 'yes', 'y') THEN TRUE
      ELSE FALSE
    END AS fl_ramo_previdencia,
    CASE
      WHEN LOWER(NULLIF(TRIM(CAST(fl_ramo_saude AS STRING)), 'null')) IN ('true', '1', 'sim', 's', 'yes', 'y') THEN TRUE
      ELSE FALSE
    END AS fl_ramo_saude,
    NULLIF(TRIM(CAST(ds_criterio_classificacao_mercado AS STRING)), 'null') AS ds_criterio_classificacao_mercado,

    CAST(COALESCE(TRY_CAST(REPLACE(NULLIF(TRIM(CAST(vl_premio_direto AS STRING)), 'null'), ',', '.') AS DECIMAL(38,2)), 0) AS DECIMAL(38,2)) AS vl_premio_direto,
    CAST(COALESCE(TRY_CAST(REPLACE(NULLIF(TRIM(CAST(vl_premio_de_seguros AS STRING)), 'null'), ',', '.') AS DECIMAL(38,2)), 0) AS DECIMAL(38,2)) AS vl_premio_de_seguros,
    CAST(COALESCE(TRY_CAST(REPLACE(NULLIF(TRIM(CAST(vl_premio_retido AS STRING)), 'null'), ',', '.') AS DECIMAL(38,2)), 0) AS DECIMAL(38,2)) AS vl_premio_retido,
    CAST(COALESCE(TRY_CAST(REPLACE(NULLIF(TRIM(CAST(vl_premio_ganho AS STRING)), 'null'), ',', '.') AS DECIMAL(38,2)), 0) AS DECIMAL(38,2)) AS vl_premio_ganho,
    CAST(COALESCE(TRY_CAST(REPLACE(NULLIF(TRIM(CAST(vl_sinistro_direto AS STRING)), 'null'), ',', '.') AS DECIMAL(38,2)), 0) AS DECIMAL(38,2)) AS vl_sinistro_direto,
    CAST(COALESCE(TRY_CAST(REPLACE(NULLIF(TRIM(CAST(vl_sinistro_retido AS STRING)), 'null'), ',', '.') AS DECIMAL(38,2)), 0) AS DECIMAL(38,2)) AS vl_sinistro_retido,
    CAST(COALESCE(TRY_CAST(REPLACE(NULLIF(TRIM(CAST(vl_despesa_comercializacao AS STRING)), 'null'), ',', '.') AS DECIMAL(38,2)), 0) AS DECIMAL(38,2)) AS vl_despesa_comercializacao,
    CAST(COALESCE(TRY_CAST(REPLACE(NULLIF(TRIM(CAST(vl_premio_emitido AS STRING)), 'null'), ',', '.') AS DECIMAL(38,2)), 0) AS DECIMAL(38,2)) AS vl_premio_emitido,
    CAST(COALESCE(TRY_CAST(REPLACE(NULLIF(TRIM(CAST(vl_premio_emitido_capitalizacao AS STRING)), 'null'), ',', '.') AS DECIMAL(38,2)), 0) AS DECIMAL(38,2)) AS vl_premio_emitido_capitalizacao,
    CAST(COALESCE(TRY_CAST(REPLACE(NULLIF(TRIM(CAST(vl_despesa_resseguro AS STRING)), 'null'), ',', '.') AS DECIMAL(38,2)), 0) AS DECIMAL(38,2)) AS vl_despesa_resseguro,
    CAST(COALESCE(TRY_CAST(REPLACE(NULLIF(TRIM(CAST(vl_sinistro_ocorrido AS STRING)), 'null'), ',', '.') AS DECIMAL(38,2)), 0) AS DECIMAL(38,2)) AS vl_sinistro_ocorrido,
    CAST(COALESCE(TRY_CAST(REPLACE(NULLIF(TRIM(CAST(vl_receita_resseguro AS STRING)), 'null'), ',', '.') AS DECIMAL(38,2)), 0) AS DECIMAL(38,2)) AS vl_receita_resseguro,
    CAST(COALESCE(TRY_CAST(REPLACE(NULLIF(TRIM(CAST(vl_sinistro_ocorrido_capitalizacao AS STRING)), 'null'), ',', '.') AS DECIMAL(38,2)), 0) AS DECIMAL(38,2)) AS vl_sinistro_ocorrido_capitalizacao,
    CAST(COALESCE(TRY_CAST(REPLACE(NULLIF(TRIM(CAST(vl_recuperacao_sinistro_ocorrido_capitalizacao AS STRING)), 'null'), ',', '.') AS DECIMAL(38,2)), 0) AS DECIMAL(38,2)) AS vl_recuperacao_sinistro_ocorrido_capitalizacao,
    CAST(COALESCE(TRY_CAST(REPLACE(NULLIF(TRIM(CAST(vl_rvne AS STRING)), 'null'), ',', '.') AS DECIMAL(38,2)), 0) AS DECIMAL(38,2)) AS vl_rvne,
    CAST(COALESCE(TRY_CAST(REPLACE(NULLIF(TRIM(CAST(vl_convenio_dpvat AS STRING)), 'null'), ',', '.') AS DECIMAL(38,2)), 0) AS DECIMAL(38,2)) AS vl_convenio_dpvat,
    CAST(COALESCE(TRY_CAST(REPLACE(NULLIF(TRIM(CAST(vl_consorcio_fundos AS STRING)), 'null'), ',', '.') AS DECIMAL(38,2)), 0) AS DECIMAL(38,2)) AS vl_consorcio_fundos,

    NULLIF(TRIM(CAST(nm_empresa_consolidada_original AS STRING)), 'null') AS nm_empresa_consolidada_original,
    NULLIF(TRIM(CAST(nm_grupo_concorrencia_n1 AS STRING)), 'null') AS nm_grupo_concorrencia_n1,
    NULLIF(TRIM(CAST(nm_grupo_concorrencia_n2 AS STRING)), 'null') AS nm_grupo_concorrencia_n2,
    NULLIF(TRIM(CAST(ds_tipo_grupo AS STRING)), 'null') AS ds_tipo_grupo,
    NULLIF(TRIM(CAST(ds_regra_mapeamento_empresa AS STRING)), 'null') AS ds_regra_mapeamento_empresa,
    NULLIF(TRIM(CAST(ds_confianca_mapeamento_empresa AS STRING)), 'null') AS ds_confianca_mapeamento_empresa,
    NULLIF(TRIM(CAST(ds_observacao_mapeamento_empresa AS STRING)), 'null') AS ds_observacao_mapeamento_empresa,
    CASE
      WHEN LOWER(NULLIF(TRIM(CAST(fl_alterou_vs_consulta_atual AS STRING)), 'null')) IN ('true', '1', 'sim', 's', 'yes', 'y') THEN TRUE
      ELSE FALSE
    END AS fl_alterou_mapeamento_empresa,

    CASE
      WHEN TRY_CAST(dt_inicio_vigencia_mapeamento_empresa AS DATE) IS NOT NULL THEN TRY_CAST(dt_inicio_vigencia_mapeamento_empresa AS DATE)
      WHEN TRY_CAST(REGEXP_REPLACE(NULLIF(TRIM(CAST(dt_inicio_vigencia_mapeamento_empresa AS STRING)), 'null'), '\\.0$', '') AS INT) IS NOT NULL
        THEN DATE_ADD(DATE '1899-12-30', TRY_CAST(REGEXP_REPLACE(NULLIF(TRIM(CAST(dt_inicio_vigencia_mapeamento_empresa AS STRING)), 'null'), '\\.0$', '') AS INT))
      ELSE DATE '1900-01-01'
    END AS dt_inicio_vigencia_empresa,
    CASE
      WHEN TRY_CAST(dt_fim_vigencia_mapeamento_empresa AS DATE) IS NOT NULL THEN TRY_CAST(dt_fim_vigencia_mapeamento_empresa AS DATE)
      WHEN TRY_CAST(REGEXP_REPLACE(NULLIF(TRIM(CAST(dt_fim_vigencia_mapeamento_empresa AS STRING)), 'null'), '\\.0$', '') AS INT) IS NOT NULL
        THEN DATE_ADD(DATE '1899-12-30', TRY_CAST(REGEXP_REPLACE(NULLIF(TRIM(CAST(dt_fim_vigencia_mapeamento_empresa AS STRING)), 'null'), '\\.0$', '') AS INT))
      ELSE NULL
    END AS dt_fim_vigencia_empresa,
    CASE
      WHEN LOWER(NULLIF(TRIM(CAST(fl_ativo_mapeamento_empresa AS STRING)), 'null')) IN ('true', '1', 'sim', 's', 'yes', 'y') THEN TRUE
      WHEN LOWER(NULLIF(TRIM(CAST(fl_ativo_mapeamento_empresa AS STRING)), 'null')) IN ('false', '0', 'nao', 'não', 'n', 'no') THEN FALSE
      ELSE TRUE
    END AS fl_ativo_mapeamento_empresa,
    NULLIF(TRIM(CAST(nr_versao_mapeamento_empresa AS STRING)), 'null') AS nr_versao_empresa,

    NULLIF(TRIM(CAST(ds_justificativa_tratamento_ramo AS STRING)), 'null') AS ds_justificativa_tratamento_ramo,
    NULLIF(TRIM(CAST(ds_fonte_regra_mapeamento_ramo AS STRING)), 'null') AS ds_fonte_regra_mapeamento_ramo,
    CASE
      WHEN TRY_CAST(dt_inicio_vigencia_mapeamento_ramo AS DATE) IS NOT NULL THEN TRY_CAST(dt_inicio_vigencia_mapeamento_ramo AS DATE)
      WHEN TRY_CAST(REGEXP_REPLACE(NULLIF(TRIM(CAST(dt_inicio_vigencia_mapeamento_ramo AS STRING)), 'null'), '\\.0$', '') AS INT) IS NOT NULL
        THEN DATE_ADD(DATE '1899-12-30', TRY_CAST(REGEXP_REPLACE(NULLIF(TRIM(CAST(dt_inicio_vigencia_mapeamento_ramo AS STRING)), 'null'), '\\.0$', '') AS INT))
      ELSE DATE '1900-01-01'
    END AS dt_inicio_vigencia_ramo,
    CASE
      WHEN TRY_CAST(dt_fim_vigencia_mapeamento_ramo AS DATE) IS NOT NULL THEN TRY_CAST(dt_fim_vigencia_mapeamento_ramo AS DATE)
      WHEN TRY_CAST(REGEXP_REPLACE(NULLIF(TRIM(CAST(dt_fim_vigencia_mapeamento_ramo AS STRING)), 'null'), '\\.0$', '') AS INT) IS NOT NULL
        THEN DATE_ADD(DATE '1899-12-30', TRY_CAST(REGEXP_REPLACE(NULLIF(TRIM(CAST(dt_fim_vigencia_mapeamento_ramo AS STRING)), 'null'), '\\.0$', '') AS INT))
      ELSE NULL
    END AS dt_fim_vigencia_ramo,
    CASE
      WHEN LOWER(NULLIF(TRIM(CAST(fl_ativo_mapeamento_ramo AS STRING)), 'null')) IN ('true', '1', 'sim', 's', 'yes', 'y') THEN TRUE
      WHEN LOWER(NULLIF(TRIM(CAST(fl_ativo_mapeamento_ramo AS STRING)), 'null')) IN ('false', '0', 'nao', 'não', 'n', 'no') THEN FALSE
      ELSE TRUE
    END AS fl_ativo_mapeamento_ramo,
    NULLIF(TRIM(CAST(nr_versao_mapeamento_ramo AS STRING)), 'null') AS nr_versao_ramo
  FROM src
)
SELECT
  nr_ano_mes_referencia,
  COALESCE(nr_ano_referencia, TRY_CAST(SUBSTR(CAST(nr_ano_mes_referencia AS STRING), 1, 4) AS INT)) AS nr_ano_referencia,
  COALESCE(nr_mes_referencia, TRY_CAST(SUBSTR(CAST(nr_ano_mes_referencia AS STRING), 5, 2) AS INT)) AS nr_mes_referencia,
  COALESCE(dt_referencia, MAKE_DATE(TRY_CAST(SUBSTR(CAST(nr_ano_mes_referencia AS STRING), 1, 4) AS INT), TRY_CAST(SUBSTR(CAST(nr_ano_mes_referencia AS STRING), 5, 2) AS INT), 1)) AS dt_referencia,
  cd_entidade_susep,
  nm_entidade_susep,
  cd_grupo_empresa_susep_original,
  cd_grupo_empresa_susep_mapeado,
  nm_grupo_empresa_susep_mapeado,
  cd_grupo_ramo_susep,
  nm_grupo_ramo_susep,
  cd_ramo_susep,
  nm_ramo_susep,
  COALESCE(cd_ramo_tratado, cd_ramo_susep) AS cd_ramo_tratado,
  COALESCE(nm_ramo_tratado, nm_ramo_susep) AS nm_ramo_tratado,
  ds_categoria_mercado_susep,
  fl_base_analitica_seguros,
  fl_ramo_dpvat,
  fl_ramo_acumulacao,
  fl_ramo_previdencia,
  fl_ramo_saude,
  ds_criterio_classificacao_mercado,
  vl_premio_direto,
  vl_premio_de_seguros,
  vl_premio_retido,
  vl_premio_ganho,
  vl_sinistro_direto,
  vl_sinistro_retido,
  vl_despesa_comercializacao,
  vl_premio_emitido,
  vl_premio_emitido_capitalizacao,
  vl_despesa_resseguro,
  vl_sinistro_ocorrido,
  vl_receita_resseguro,
  vl_sinistro_ocorrido_capitalizacao,
  vl_recuperacao_sinistro_ocorrido_capitalizacao,
  vl_rvne,
  vl_convenio_dpvat,
  vl_consorcio_fundos,
  nm_empresa_consolidada_original,
  nm_grupo_concorrencia_n1,
  nm_grupo_concorrencia_n2,
  ds_tipo_grupo,
  ds_regra_mapeamento_empresa,
  ds_confianca_mapeamento_empresa,
  ds_observacao_mapeamento_empresa,
  fl_alterou_mapeamento_empresa,
  dt_inicio_vigencia_empresa,
  COALESCE(dt_fim_vigencia_empresa, DATE '9999-12-31') AS dt_fim_vigencia_empresa,
  CASE WHEN fl_ativo_mapeamento_empresa = TRUE AND dt_fim_vigencia_empresa IS NULL THEN TRUE ELSE fl_ativo_mapeamento_empresa END AS fl_registro_atual_empresa,
  COALESCE(nr_versao_empresa, 'v1') AS nr_versao_empresa,
  ds_justificativa_tratamento_ramo,
  ds_fonte_regra_mapeamento_ramo,
  dt_inicio_vigencia_ramo,
  COALESCE(dt_fim_vigencia_ramo, DATE '9999-12-31') AS dt_fim_vigencia_ramo,
  CASE WHEN fl_ativo_mapeamento_ramo = TRUE AND dt_fim_vigencia_ramo IS NULL THEN TRUE ELSE fl_ativo_mapeamento_ramo END AS fl_registro_atual_ramo,
  COALESCE(nr_versao_ramo, 'v1') AS nr_versao_ramo
FROM typed
WHERE nr_ano_mes_referencia IS NOT NULL
  AND cd_entidade_susep IS NOT NULL
  AND COALESCE(cd_ramo_tratado, cd_ramo_susep) IS NOT NULL
  AND fl_base_analitica_seguros = TRUE;

SELECT
  COUNT(*) AS qt_linhas_base_gold,
  COUNT(DISTINCT nr_ano_mes_referencia) AS qt_periodos,
  COUNT(DISTINCT cd_entidade_susep) AS qt_entidades,
  COUNT(DISTINCT cd_ramo_tratado) AS qt_ramos_tratados
FROM vw_gold_base_seguro_mensal;


In [0]:

-- ============================================================
-- 5. LOAD GOLD DIMENSIONS AND FACT TABLE
-- ============================================================
-- Full refresh load strategy.
-- The same deterministic SK expressions are used in dimensions and fact.

INSERT OVERWRITE TABLE susep_gold.tb_dim_periodo (
  sk_periodo,
  nr_ano_mes_referencia,
  nr_ano_referencia,
  nr_mes_referencia,
  dt_referencia,
  nm_mes_referencia,
  sg_mes_referencia,
  nr_trimestre_referencia,
  nm_trimestre_referencia,
  nr_semestre_referencia,
  nm_semestre_referencia
)
SELECT DISTINCT
  nr_ano_mes_referencia AS sk_periodo,
  nr_ano_mes_referencia,
  nr_ano_referencia,
  nr_mes_referencia,
  dt_referencia,
  CASE nr_mes_referencia
    WHEN 1 THEN 'Janeiro'
    WHEN 2 THEN 'Fevereiro'
    WHEN 3 THEN 'Março'
    WHEN 4 THEN 'Abril'
    WHEN 5 THEN 'Maio'
    WHEN 6 THEN 'Junho'
    WHEN 7 THEN 'Julho'
    WHEN 8 THEN 'Agosto'
    WHEN 9 THEN 'Setembro'
    WHEN 10 THEN 'Outubro'
    WHEN 11 THEN 'Novembro'
    WHEN 12 THEN 'Dezembro'
  END AS nm_mes_referencia,
  CASE nr_mes_referencia
    WHEN 1 THEN 'Jan'
    WHEN 2 THEN 'Fev'
    WHEN 3 THEN 'Mar'
    WHEN 4 THEN 'Abr'
    WHEN 5 THEN 'Mai'
    WHEN 6 THEN 'Jun'
    WHEN 7 THEN 'Jul'
    WHEN 8 THEN 'Ago'
    WHEN 9 THEN 'Set'
    WHEN 10 THEN 'Out'
    WHEN 11 THEN 'Nov'
    WHEN 12 THEN 'Dez'
  END AS sg_mes_referencia,
  CAST(CEIL(nr_mes_referencia / 3.0) AS INT) AS nr_trimestre_referencia,
  CONCAT(CAST(CAST(CEIL(nr_mes_referencia / 3.0) AS INT) AS STRING), 'º Trimestre') AS nm_trimestre_referencia,
  CASE WHEN nr_mes_referencia <= 6 THEN 1 ELSE 2 END AS nr_semestre_referencia,
  CASE WHEN nr_mes_referencia <= 6 THEN '1º Semestre' ELSE '2º Semestre' END AS nm_semestre_referencia
FROM vw_gold_base_seguro_mensal;

INSERT OVERWRITE TABLE susep_gold.tb_dim_empresa_susep (
  sk_empresa_susep,
  cd_entidade_susep,
  nm_entidade_susep,
  cd_grupo_empresa_susep_original,
  cd_grupo_empresa_susep_mapeado,
  nm_grupo_empresa_susep_mapeado,
  nm_empresa_consolidada_original,
  nm_grupo_concorrencia_n1,
  nm_grupo_concorrencia_n2,
  ds_tipo_grupo,
  ds_regra_mapeamento_empresa,
  ds_confianca_mapeamento_empresa,
  ds_observacao_mapeamento_empresa,
  fl_alterou_mapeamento_empresa,
  dt_inicio_vigencia_empresa,
  dt_fim_vigencia_empresa,
  fl_registro_atual,
  nr_versao_empresa
)
WITH empresas AS (
  SELECT
    ABS(XXHASH64(
      COALESCE(cd_entidade_susep, '-1'),
      COALESCE(CAST(dt_inicio_vigencia_empresa AS STRING), '1900-01-01'),
      COALESCE(CAST(dt_fim_vigencia_empresa AS STRING), '9999-12-31'),
      COALESCE(nr_versao_empresa, 'v1')
    )) AS sk_empresa_susep,
    cd_entidade_susep,
    nm_entidade_susep,
    cd_grupo_empresa_susep_original,
    cd_grupo_empresa_susep_mapeado,
    nm_grupo_empresa_susep_mapeado,
    nm_empresa_consolidada_original,
    nm_grupo_concorrencia_n1,
    nm_grupo_concorrencia_n2,
    ds_tipo_grupo,
    ds_regra_mapeamento_empresa,
    ds_confianca_mapeamento_empresa,
    ds_observacao_mapeamento_empresa,
    fl_alterou_mapeamento_empresa,
    dt_inicio_vigencia_empresa,
    dt_fim_vigencia_empresa,
    fl_registro_atual_empresa AS fl_registro_atual,
    nr_versao_empresa,
    ROW_NUMBER() OVER (
      PARTITION BY cd_entidade_susep, dt_inicio_vigencia_empresa, dt_fim_vigencia_empresa, nr_versao_empresa
      ORDER BY fl_registro_atual_empresa DESC, nm_entidade_susep, nm_grupo_concorrencia_n1
    ) AS nr_linha
  FROM vw_gold_base_seguro_mensal
)
SELECT
  sk_empresa_susep,
  cd_entidade_susep,
  nm_entidade_susep,
  cd_grupo_empresa_susep_original,
  cd_grupo_empresa_susep_mapeado,
  nm_grupo_empresa_susep_mapeado,
  nm_empresa_consolidada_original,
  nm_grupo_concorrencia_n1,
  nm_grupo_concorrencia_n2,
  ds_tipo_grupo,
  ds_regra_mapeamento_empresa,
  ds_confianca_mapeamento_empresa,
  ds_observacao_mapeamento_empresa,
  fl_alterou_mapeamento_empresa,
  dt_inicio_vigencia_empresa,
  dt_fim_vigencia_empresa,
  fl_registro_atual,
  nr_versao_empresa
FROM empresas
WHERE nr_linha = 1;

INSERT OVERWRITE TABLE susep_gold.tb_dim_ramo_susep (
  sk_ramo_susep,
  cd_grupo_ramo_susep,
  nm_grupo_ramo_susep,
  cd_ramo_susep,
  nm_ramo_susep,
  cd_ramo_tratado,
  nm_ramo_tratado,
  ds_categoria_mercado_susep,
  fl_base_analitica_seguros,
  fl_ramo_dpvat,
  fl_ramo_acumulacao,
  fl_ramo_previdencia,
  fl_ramo_saude,
  ds_criterio_classificacao_mercado,
  ds_justificativa_tratamento_ramo,
  ds_fonte_regra_mapeamento_ramo,
  dt_inicio_vigencia_ramo,
  dt_fim_vigencia_ramo,
  fl_registro_atual,
  nr_versao_ramo
)
WITH ramos AS (
  SELECT
    ABS(XXHASH64(
      COALESCE(cd_ramo_susep, '-1'),
      COALESCE(cd_ramo_tratado, '-1'),
      COALESCE(CAST(dt_inicio_vigencia_ramo AS STRING), '1900-01-01'),
      COALESCE(CAST(dt_fim_vigencia_ramo AS STRING), '9999-12-31'),
      COALESCE(nr_versao_ramo, 'v1')
    )) AS sk_ramo_susep,
    cd_grupo_ramo_susep,
    nm_grupo_ramo_susep,
    cd_ramo_susep,
    nm_ramo_susep,
    cd_ramo_tratado,
    nm_ramo_tratado,
    ds_categoria_mercado_susep,
    fl_base_analitica_seguros,
    fl_ramo_dpvat,
    fl_ramo_acumulacao,
    fl_ramo_previdencia,
    fl_ramo_saude,
    ds_criterio_classificacao_mercado,
    ds_justificativa_tratamento_ramo,
    ds_fonte_regra_mapeamento_ramo,
    dt_inicio_vigencia_ramo,
    dt_fim_vigencia_ramo,
    fl_registro_atual_ramo AS fl_registro_atual,
    nr_versao_ramo,
    ROW_NUMBER() OVER (
      PARTITION BY cd_ramo_susep, cd_ramo_tratado, dt_inicio_vigencia_ramo, dt_fim_vigencia_ramo, nr_versao_ramo
      ORDER BY fl_registro_atual_ramo DESC, nm_ramo_tratado, nm_ramo_susep
    ) AS nr_linha
  FROM vw_gold_base_seguro_mensal
)
SELECT
  sk_ramo_susep,
  cd_grupo_ramo_susep,
  nm_grupo_ramo_susep,
  cd_ramo_susep,
  nm_ramo_susep,
  cd_ramo_tratado,
  nm_ramo_tratado,
  ds_categoria_mercado_susep,
  fl_base_analitica_seguros,
  fl_ramo_dpvat,
  fl_ramo_acumulacao,
  fl_ramo_previdencia,
  fl_ramo_saude,
  ds_criterio_classificacao_mercado,
  ds_justificativa_tratamento_ramo,
  ds_fonte_regra_mapeamento_ramo,
  dt_inicio_vigencia_ramo,
  dt_fim_vigencia_ramo,
  fl_registro_atual,
  nr_versao_ramo
FROM ramos
WHERE nr_linha = 1;

INSERT OVERWRITE TABLE susep_gold.tb_fato_seguro_mensal_susep (
  sk_periodo,
  sk_empresa_susep,
  sk_ramo_susep,
  vl_premio_direto,
  vl_premio_de_seguros,
  vl_premio_retido,
  vl_premio_ganho,
  vl_sinistro_direto,
  vl_sinistro_retido,
  vl_despesa_comercializacao,
  vl_premio_emitido,
  vl_premio_emitido_capitalizacao,
  vl_despesa_resseguro,
  vl_sinistro_ocorrido,
  vl_receita_resseguro,
  vl_sinistro_ocorrido_capitalizacao,
  vl_recuperacao_sinistro_ocorrido_capitalizacao,
  vl_rvne,
  vl_convenio_dpvat,
  vl_consorcio_fundos
)
SELECT
  nr_ano_mes_referencia AS sk_periodo,
  ABS(XXHASH64(
    COALESCE(cd_entidade_susep, '-1'),
    COALESCE(CAST(dt_inicio_vigencia_empresa AS STRING), '1900-01-01'),
    COALESCE(CAST(dt_fim_vigencia_empresa AS STRING), '9999-12-31'),
    COALESCE(nr_versao_empresa, 'v1')
  )) AS sk_empresa_susep,
  ABS(XXHASH64(
    COALESCE(cd_ramo_susep, '-1'),
    COALESCE(cd_ramo_tratado, '-1'),
    COALESCE(CAST(dt_inicio_vigencia_ramo AS STRING), '1900-01-01'),
    COALESCE(CAST(dt_fim_vigencia_ramo AS STRING), '9999-12-31'),
    COALESCE(nr_versao_ramo, 'v1')
  )) AS sk_ramo_susep,
  SUM(vl_premio_direto) AS vl_premio_direto,
  SUM(vl_premio_de_seguros) AS vl_premio_de_seguros,
  SUM(vl_premio_retido) AS vl_premio_retido,
  SUM(vl_premio_ganho) AS vl_premio_ganho,
  SUM(vl_sinistro_direto) AS vl_sinistro_direto,
  SUM(vl_sinistro_retido) AS vl_sinistro_retido,
  SUM(vl_despesa_comercializacao) AS vl_despesa_comercializacao,
  SUM(vl_premio_emitido) AS vl_premio_emitido,
  SUM(vl_premio_emitido_capitalizacao) AS vl_premio_emitido_capitalizacao,
  SUM(vl_despesa_resseguro) AS vl_despesa_resseguro,
  SUM(vl_sinistro_ocorrido) AS vl_sinistro_ocorrido,
  SUM(vl_receita_resseguro) AS vl_receita_resseguro,
  SUM(vl_sinistro_ocorrido_capitalizacao) AS vl_sinistro_ocorrido_capitalizacao,
  SUM(vl_recuperacao_sinistro_ocorrido_capitalizacao) AS vl_recuperacao_sinistro_ocorrido_capitalizacao,
  SUM(vl_rvne) AS vl_rvne,
  SUM(vl_convenio_dpvat) AS vl_convenio_dpvat,
  SUM(vl_consorcio_fundos) AS vl_consorcio_fundos
FROM vw_gold_base_seguro_mensal
GROUP BY
  nr_ano_mes_referencia,
  ABS(XXHASH64(
    COALESCE(cd_entidade_susep, '-1'),
    COALESCE(CAST(dt_inicio_vigencia_empresa AS STRING), '1900-01-01'),
    COALESCE(CAST(dt_fim_vigencia_empresa AS STRING), '9999-12-31'),
    COALESCE(nr_versao_empresa, 'v1')
  )),
  ABS(XXHASH64(
    COALESCE(cd_ramo_susep, '-1'),
    COALESCE(cd_ramo_tratado, '-1'),
    COALESCE(CAST(dt_inicio_vigencia_ramo AS STRING), '1900-01-01'),
    COALESCE(CAST(dt_fim_vigencia_ramo AS STRING), '9999-12-31'),
    COALESCE(nr_versao_ramo, 'v1')
  ));


In [0]:

-- ============================================================
-- 6. CREATE ANALYTICAL VIEWS
-- ============================================================
-- These views keep the physical model dimensional and make consumption easier
-- for SQL exploration, dashboards and Power BI semantic models.

/*CREATE OR REPLACE VIEW susep_gold.vw_fato_seguro_mensal_susep AS
SELECT
  f.sk_periodo,
  f.sk_empresa_susep,
  f.sk_ramo_susep,
  p.nr_ano_mes_referencia,
  p.nr_ano_referencia,
  p.nr_mes_referencia,
  p.dt_referencia,
  p.nm_mes_referencia,
  p.sg_mes_referencia,
  p.nr_trimestre_referencia,
  p.nm_trimestre_referencia,
  p.nr_semestre_referencia,
  p.nm_semestre_referencia,
  e.cd_entidade_susep,
  e.nm_entidade_susep,
  e.cd_grupo_empresa_susep_original,
  e.cd_grupo_empresa_susep_mapeado,
  e.nm_grupo_empresa_susep_mapeado,
  e.nm_empresa_consolidada_original,
  e.nm_grupo_concorrencia_n1,
  e.nm_grupo_concorrencia_n2,
  e.ds_tipo_grupo,
  r.cd_grupo_ramo_susep,
  r.nm_grupo_ramo_susep,
  r.cd_ramo_susep,
  r.nm_ramo_susep,
  r.cd_ramo_tratado,
  r.nm_ramo_tratado,
  r.ds_categoria_mercado_susep,
  r.fl_base_analitica_seguros,
  r.fl_ramo_dpvat,
  r.fl_ramo_acumulacao,
  r.fl_ramo_previdencia,
  r.fl_ramo_saude,
  r.ds_criterio_classificacao_mercado,
  f.vl_premio_direto,
  f.vl_premio_de_seguros,
  f.vl_premio_retido,
  f.vl_premio_ganho,
  f.vl_sinistro_direto,
  f.vl_sinistro_retido,
  f.vl_despesa_comercializacao,
  f.vl_premio_emitido,
  f.vl_premio_emitido_capitalizacao,
  f.vl_despesa_resseguro,
  f.vl_sinistro_ocorrido,
  f.vl_receita_resseguro,
  f.vl_sinistro_ocorrido_capitalizacao,
  f.vl_recuperacao_sinistro_ocorrido_capitalizacao,
  f.vl_rvne,
  f.vl_convenio_dpvat,
  f.vl_consorcio_fundos
FROM susep_gold.tb_fato_seguro_mensal_susep f
LEFT JOIN susep_gold.tb_dim_periodo p
  ON f.sk_periodo = p.sk_periodo
LEFT JOIN susep_gold.tb_dim_empresa_susep e
  ON f.sk_empresa_susep = e.sk_empresa_susep
LEFT JOIN susep_gold.tb_dim_ramo_susep r
  ON f.sk_ramo_susep = r.sk_ramo_susep;

CREATE OR REPLACE VIEW susep_gold.vw_fato_seguro_mensal_susep_sem_dpvat AS
SELECT *
FROM susep_gold.vw_fato_seguro_mensal_susep
WHERE fl_ramo_dpvat = FALSE;
*/

In [0]:

-- ============================================================
-- 7. ADD UNITY CATALOG TAGS
-- ============================================================
-- Execute this step in Unity Catalog enabled workspaces.
-- The table and column comments were already defined in the CREATE TABLE step.

ALTER TABLE susep_gold.tb_dim_periodo SET TAGS (
  'camada' = 'gold',
  'dominio_dado' = 'seguros_susep',
  'tipo_tabela' = 'dimensao',
  'granularidade' = 'mes_referencia',
  'projeto' = 'lakehouse_susep',
  'classificacao' = 'publico'
);

ALTER TABLE susep_gold.tb_dim_empresa_susep SET TAGS (
  'camada' = 'gold',
  'dominio_dado' = 'seguros_susep',
  'tipo_tabela' = 'dimensao_scd',
  'granularidade' = 'entidade_susep_versao_scd',
  'projeto' = 'lakehouse_susep',
  'classificacao' = 'publico'
);

ALTER TABLE susep_gold.tb_dim_ramo_susep SET TAGS (
  'camada' = 'gold',
  'dominio_dado' = 'seguros_susep',
  'tipo_tabela' = 'dimensao_scd',
  'granularidade' = 'ramo_susep_tratado_versao_scd',
  'projeto' = 'lakehouse_susep',
  'classificacao' = 'publico'
);

ALTER TABLE susep_gold.tb_fato_seguro_mensal_susep SET TAGS (
  'camada' = 'gold',
  'dominio_dado' = 'seguros_susep',
  'tipo_tabela' = 'fato',
  'granularidade' = 'periodo_empresa_ramo',
  'projeto' = 'lakehouse_susep',
  'classificacao' = 'publico'
);

ALTER TABLE susep_gold.tb_dim_periodo ALTER COLUMN sk_periodo SET TAGS ('tipo_coluna' = 'surrogate_key', 'dominio_dado' = 'temporal');
ALTER TABLE susep_gold.tb_dim_periodo ALTER COLUMN nr_ano_mes_referencia SET TAGS ('tipo_coluna' = 'chave_analitica', 'dominio_dado' = 'temporal');
ALTER TABLE susep_gold.tb_dim_periodo ALTER COLUMN dt_referencia SET TAGS ('tipo_coluna' = 'atributo', 'dominio_dado' = 'temporal');

ALTER TABLE susep_gold.tb_dim_empresa_susep ALTER COLUMN sk_empresa_susep SET TAGS ('tipo_coluna' = 'surrogate_key', 'dominio_dado' = 'empresa');
ALTER TABLE susep_gold.tb_dim_empresa_susep ALTER COLUMN cd_entidade_susep SET TAGS ('tipo_coluna' = 'business_key', 'dominio_dado' = 'empresa');
ALTER TABLE susep_gold.tb_dim_empresa_susep ALTER COLUMN nm_entidade_susep SET TAGS ('tipo_coluna' = 'atributo', 'dominio_dado' = 'empresa');
ALTER TABLE susep_gold.tb_dim_empresa_susep ALTER COLUMN nm_grupo_concorrencia_n1 SET TAGS ('tipo_coluna' = 'atributo_analitico', 'dominio_dado' = 'empresa');
ALTER TABLE susep_gold.tb_dim_empresa_susep ALTER COLUMN dt_inicio_vigencia_empresa SET TAGS ('tipo_coluna' = 'scd', 'dominio_dado' = 'empresa');
ALTER TABLE susep_gold.tb_dim_empresa_susep ALTER COLUMN dt_fim_vigencia_empresa SET TAGS ('tipo_coluna' = 'scd', 'dominio_dado' = 'empresa');
ALTER TABLE susep_gold.tb_dim_empresa_susep ALTER COLUMN fl_registro_atual SET TAGS ('tipo_coluna' = 'scd', 'dominio_dado' = 'empresa');

ALTER TABLE susep_gold.tb_dim_ramo_susep ALTER COLUMN sk_ramo_susep SET TAGS ('tipo_coluna' = 'surrogate_key', 'dominio_dado' = 'produto');
ALTER TABLE susep_gold.tb_dim_ramo_susep ALTER COLUMN cd_ramo_susep SET TAGS ('tipo_coluna' = 'business_key', 'dominio_dado' = 'produto');
ALTER TABLE susep_gold.tb_dim_ramo_susep ALTER COLUMN cd_ramo_tratado SET TAGS ('tipo_coluna' = 'business_key_tratada', 'dominio_dado' = 'produto');
ALTER TABLE susep_gold.tb_dim_ramo_susep ALTER COLUMN ds_categoria_mercado_susep SET TAGS ('tipo_coluna' = 'classificacao', 'dominio_dado' = 'mercado_susep');
ALTER TABLE susep_gold.tb_dim_ramo_susep ALTER COLUMN fl_base_analitica_seguros SET TAGS ('tipo_coluna' = 'flag_analitica', 'dominio_dado' = 'mercado_susep');
ALTER TABLE susep_gold.tb_dim_ramo_susep ALTER COLUMN dt_inicio_vigencia_ramo SET TAGS ('tipo_coluna' = 'scd', 'dominio_dado' = 'produto');
ALTER TABLE susep_gold.tb_dim_ramo_susep ALTER COLUMN dt_fim_vigencia_ramo SET TAGS ('tipo_coluna' = 'scd', 'dominio_dado' = 'produto');
ALTER TABLE susep_gold.tb_dim_ramo_susep ALTER COLUMN fl_registro_atual SET TAGS ('tipo_coluna' = 'scd', 'dominio_dado' = 'produto');

ALTER TABLE susep_gold.tb_fato_seguro_mensal_susep ALTER COLUMN sk_periodo SET TAGS ('tipo_coluna' = 'foreign_key', 'dominio_dado' = 'temporal');
ALTER TABLE susep_gold.tb_fato_seguro_mensal_susep ALTER COLUMN sk_empresa_susep SET TAGS ('tipo_coluna' = 'foreign_key', 'dominio_dado' = 'empresa');
ALTER TABLE susep_gold.tb_fato_seguro_mensal_susep ALTER COLUMN sk_ramo_susep SET TAGS ('tipo_coluna' = 'foreign_key', 'dominio_dado' = 'produto');
ALTER TABLE susep_gold.tb_fato_seguro_mensal_susep ALTER COLUMN vl_premio_direto SET TAGS ('tipo_coluna' = 'medida', 'dominio_dado' = 'financeiro');
ALTER TABLE susep_gold.tb_fato_seguro_mensal_susep ALTER COLUMN vl_premio_ganho SET TAGS ('tipo_coluna' = 'medida', 'dominio_dado' = 'financeiro');
ALTER TABLE susep_gold.tb_fato_seguro_mensal_susep ALTER COLUMN vl_sinistro_direto SET TAGS ('tipo_coluna' = 'medida', 'dominio_dado' = 'financeiro');
ALTER TABLE susep_gold.tb_fato_seguro_mensal_susep ALTER COLUMN vl_sinistro_ocorrido SET TAGS ('tipo_coluna' = 'medida', 'dominio_dado' = 'financeiro');


In [0]:

-- ============================================================
-- 8. POST LOAD CHECK - VOLUME AND REFERENTIAL COVERAGE
-- ============================================================

SELECT 'tb_dim_periodo' AS nm_tabela, COUNT(*) AS qt_linhas FROM susep_gold.tb_dim_periodo
UNION ALL
SELECT 'tb_dim_empresa_susep' AS nm_tabela, COUNT(*) AS qt_linhas FROM susep_gold.tb_dim_empresa_susep
UNION ALL
SELECT 'tb_dim_ramo_susep' AS nm_tabela, COUNT(*) AS qt_linhas FROM susep_gold.tb_dim_ramo_susep
UNION ALL
SELECT 'tb_fato_seguro_mensal_susep' AS nm_tabela, COUNT(*) AS qt_linhas FROM susep_gold.tb_fato_seguro_mensal_susep;

SELECT
  COUNT(*) AS qt_linhas_fato,
  COUNT_IF(p.sk_periodo IS NULL) AS qt_fato_sem_dim_periodo,
  COUNT_IF(e.sk_empresa_susep IS NULL) AS qt_fato_sem_dim_empresa,
  COUNT_IF(r.sk_ramo_susep IS NULL) AS qt_fato_sem_dim_ramo,
  MIN(p.dt_referencia) AS dt_referencia_min,
  MAX(p.dt_referencia) AS dt_referencia_max
FROM susep_gold.tb_fato_seguro_mensal_susep f
LEFT JOIN susep_gold.tb_dim_periodo p
  ON f.sk_periodo = p.sk_periodo
LEFT JOIN susep_gold.tb_dim_empresa_susep e
  ON f.sk_empresa_susep = e.sk_empresa_susep
LEFT JOIN susep_gold.tb_dim_ramo_susep r
  ON f.sk_ramo_susep = r.sk_ramo_susep;


In [0]:

-- ============================================================
-- 9. POST LOAD CHECK - MONTHLY INSURANCE VIEW
-- ============================================================

SELECT
  p.nr_ano_mes_referencia,
  COUNT(*) AS qt_linhas_fato,
  SUM(f.vl_premio_direto) AS vl_premio_direto,
  SUM(f.vl_premio_ganho) AS vl_premio_ganho,
  SUM(f.vl_sinistro_direto) AS vl_sinistro_direto,
  SUM(f.vl_sinistro_ocorrido) AS vl_sinistro_ocorrido,
  CASE
    WHEN SUM(f.vl_premio_ganho) <> 0
      THEN SUM(f.vl_sinistro_ocorrido) / SUM(f.vl_premio_ganho)
    ELSE NULL
  END AS pc_sinistralidade_ocorrida
FROM susep_gold.tb_fato_seguro_mensal_susep f
INNER JOIN susep_gold.tb_dim_periodo p
  ON f.sk_periodo = p.sk_periodo
GROUP BY p.nr_ano_mes_referencia
ORDER BY p.nr_ano_mes_referencia DESC
LIMIT 24;


In [0]:

-- ============================================================
-- 10. POST LOAD CHECK - COMPANY AND BRANCH RANKING
-- ============================================================

SELECT
  e.nm_grupo_concorrencia_n1,
  r.nm_ramo_tratado,
  SUM(f.vl_premio_direto) AS vl_premio_direto,
  SUM(f.vl_premio_ganho) AS vl_premio_ganho,
  SUM(f.vl_sinistro_ocorrido) AS vl_sinistro_ocorrido,
  CASE
    WHEN SUM(f.vl_premio_ganho) <> 0
      THEN SUM(f.vl_sinistro_ocorrido) / SUM(f.vl_premio_ganho)
    ELSE NULL
  END AS pc_sinistralidade_ocorrida
FROM susep_gold.tb_fato_seguro_mensal_susep f
INNER JOIN susep_gold.tb_dim_empresa_susep e
  ON f.sk_empresa_susep = e.sk_empresa_susep
INNER JOIN susep_gold.tb_dim_ramo_susep r
  ON f.sk_ramo_susep = r.sk_ramo_susep
GROUP BY
  e.nm_grupo_concorrencia_n1,
  r.nm_ramo_tratado
ORDER BY vl_premio_direto DESC
LIMIT 50;


In [0]:

-- ============================================================
-- 11. OPTIONAL OPTIMIZATION
-- ============================================================
-- For larger historical loads, this can improve query performance.
-- Keep this step optional for academic/MVP execution if cluster cost is a concern.

OPTIMIZE susep_gold.tb_fato_seguro_mensal_susep
ZORDER BY (sk_empresa_susep, sk_ramo_susep);

OPTIMIZE susep_gold.tb_dim_empresa_susep
ZORDER BY (cd_entidade_susep, nm_grupo_concorrencia_n1);

OPTIMIZE susep_gold.tb_dim_ramo_susep
ZORDER BY (cd_ramo_tratado, ds_categoria_mercado_susep);
